# 07 · Inverse Problems — Discovering Physical Parameters from Data

**Level:** Advanced
**Prerequisites:** `03_heat_equation`

This is where PINNs become genuinely special. In a **forward** problem we know the
equation and solve for the field. In an **inverse** problem we have some sparse,
noisy measurements and we want to discover an *unknown parameter of the physics*
— a diffusivity, a reaction rate, a material property. PINNs do this almost for
free: we simply make the unknown parameter a trainable variable and let the same
loss optimise it alongside the network weights.

### What you will learn
1. Making a physical constant a **learnable `nn.Parameter`**
2. Combining a **data loss** (measurements) with a **physics loss**
3. Recovering the true parameter from *sparse and noisy* data
4. Why the physics term acts as a powerful regulariser

### References
- Raissi, Perdikaris & Karniadakis (2019) — introduced PINN inverse problems
- Raissi, Yazdani & Karniadakis (2020), *Hidden fluid mechanics*, Science 367:1026-1030


## 1 · Setup

We generate synthetic data from the heat equation with a **true** diffusivity
$\alpha^\star = 0.4$, add noise, keep only a handful of sensor readings, then ask
the PINN to recover $\alpha$ while it simultaneously reconstructs the field.

$$\frac{\partial u}{\partial t} = \alpha\,\frac{\partial^2 u}{\partial x^2},
\qquad u(x,t)=\sin(\pi x)e^{-\alpha^\star \pi^2 t}\;(\text{ground truth}).$$

The learnable quantity is $\alpha$. We initialise it *wrong* on purpose.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

alpha_true = 0.4
def ground_truth(x, t):
    return np.sin(np.pi*x)*np.exp(-alpha_true*np.pi**2*t)

# --- Synthetic sparse, noisy sensor data ---
N_data = 40
x_d = np.random.rand(N_data)
t_d = np.random.rand(N_data)
u_d = ground_truth(x_d, t_d) + 0.02*np.random.randn(N_data)   # 2% noise
x_d = torch.tensor(x_d, dtype=torch.float32, device=device).view(-1,1)
t_d = torch.tensor(t_d, dtype=torch.float32, device=device).view(-1,1)
u_d = torch.tensor(u_d, dtype=torch.float32, device=device).view(-1,1)
print(f"{N_data} noisy sensor points")


In [ ]:
class PINN(nn.Module):
    def __init__(self, layers, alpha_init):
        super().__init__()
        seq = []
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers)-2: seq.append(nn.Tanh())
        self.net = nn.Sequential(*seq)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
        # THE UNKNOWN PARAMETER, learned jointly with the weights:
        self.log_alpha = nn.Parameter(torch.tensor(np.log(alpha_init), dtype=torch.float32))
    @property
    def alpha(self):
        return torch.exp(self.log_alpha)   # keep alpha positive
    def forward(self, x, t):
        return self.net(torch.cat([x,t],1))

model = PINN([2,40,40,40,1], alpha_init=1.5).to(device)   # start deliberately wrong
print("initial alpha guess:", model.alpha.item())


In [ ]:
# collocation points for the physics residual
N_f = 3000
x_f = torch.rand(N_f,1,device=device); t_f = torch.rand(N_f,1,device=device)
x_f.requires_grad_(True); t_f.requires_grad_(True)

def residual(x, t):
    u = model(x, t)
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx= torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return u_t - model.alpha * u_xx

def loss_fn():
    L_data = torch.mean((model(x_d, t_d) - u_d)**2)   # fit the sensors
    L_phys = torch.mean(residual(x_f, t_f)**2)         # obey the PDE
    return L_data + L_phys, L_data.item(), L_phys.item()


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
EPOCHS = 12000
alpha_hist = []
for ep in range(EPOCHS):
    opt.zero_grad(); loss, ld, lp = loss_fn(); loss.backward(); opt.step()
    alpha_hist.append(model.alpha.item())
    if ep % 2000 == 0:
        print(f"epoch {ep:5d} | loss {loss.item():.3e} | data {ld:.2e} phys {lp:.2e} | alpha {model.alpha.item():.4f}")
print(f"\nRecovered alpha = {model.alpha.item():.4f}   (true = {alpha_true})")
print(f"relative error   = {abs(model.alpha.item()-alpha_true)/alpha_true*100:.2f}%")


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(alpha_hist, lw=2)
ax[0].axhline(alpha_true, color='k', ls='--', label='true alpha')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('estimated alpha'); ax[0].legend()
ax[0].set_title('Parameter converges to the true value')

# reconstructed field vs truth
xs = np.linspace(0,1,100); ts = np.linspace(0,1,100)
X,Tg = np.meshgrid(xs,ts)
xf = torch.tensor(X.ravel(),dtype=torch.float32,device=device).view(-1,1)
tf = torch.tensor(Tg.ravel(),dtype=torch.float32,device=device).view(-1,1)
with torch.no_grad():
    U = model(xf,tf).cpu().numpy().reshape(100,100)
im = ax[1].pcolormesh(X,Tg,U,shading='auto'); ax[1].set_title('Reconstructed u(x,t)')
ax[1].scatter(x_d.cpu(), t_d.cpu(), c='r', s=15, label='sensors')
ax[1].set_xlabel('x'); ax[1].set_ylabel('t'); ax[1].legend(); fig.colorbar(im,ax=ax[1])
plt.tight_layout(); plt.show()


## 2 · Why this works so well

With only 40 noisy points, a plain regression network would overfit wildly.
The **physics loss constrains the space of admissible solutions** to those that
obey the heat equation — so the network cannot fit the noise without violating
the PDE. The unknown `alpha` is nudged to whatever value makes the data and the
physics *mutually consistent*. This coupling is the essence of PINN inverse
problems and of "hidden physics" discovery (Raissi et al. 2020).

Two practical tricks used above:
- We optimise `log(alpha)` instead of `alpha` to **guarantee positivity**.
- Data and physics losses are on comparable scales here; for real problems you
  often need to weight them (see the self-adaptive weighting literature).

## 3 · Exercises
1. Increase the noise to 10% and reduce to 15 sensors. How far can you push it?
2. Make **two** parameters unknown (e.g. a reaction-diffusion system with both a
   diffusivity and a rate constant).
3. Discover a *spatially varying* coefficient `alpha(x)` by replacing the scalar
   with a small sub-network — a step toward full field inversion.

**Next:** `08_adaptive_sampling` — putting collocation points where they matter.
